In [ ]:
print('Loading Packages...\n')
%matplotlib osx
import sys, os          
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import trange
import numpy as np
from scipy.optimize import curve_fit
import pandas as pd
from collections import defaultdict
import scipy.stats
import matplotlib.gridspec as gridspec
from scipy.stats import chi2_contingency
import random
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
import matplotlib.patches as mpatches
font = {'family' : 'serif', 'size' : 14 }
mpl.rc('font', **font)
mpl.rcParams['mathtext.fontset'] = 'cm' # Set the math font to Computer Modern
mpl.rcParams['legend.fontsize'] = 16
# Set default figure and axes face colors to white
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

#POT = 2.283e20     # MC POT (old files)
POT = 2.3e20   # new files

# tank dimensions:
radius = 1.524
half_height = 1.98

bunch_time_cutoff = 1560
time_to_prompt_end = 258       # MC approximate time from end of spill to 2us cutoff (chain last MC bunch to observed data last bunch)

# spill duration for data
spill_start_22 = 206
spill_end_22 = 1739

spill_start_23 = 189
spill_end_23 = 1719


df = pd.read_csv('../MC_GENIE_Jul_2025/FullTankPMTGeometry.csv')
channel_number = []; PMT_type = []
for i in range(len(df['channel_num'])):
    channel_number.append(df['channel_num'][i])
    PMT_type.append(df['PMT_type'][i])
    

print('done')

In [ ]:
# MC

#MC_directory = 'GENIE_CV_tilt_MC_ToolChain/'
MC_directory = '42_44/'
file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]

ALTERNATIVE_SIDEBAND = False    # an alternative sideband: muons with slightly different selection criteria

WEIGHTING = True

# scalar accumulators
MC_cluster_number    = []
MC_cluster_time      = []
MC_cluster_charge    = []
MC_cluster_Qb        = []
MC_cluster_Hits      = []
MC_TankMRDCoinc      = []
MC_NoVeto            = []
MC_Vtx               = []
MC_Vty               = []
MC_Vtz               = []
MC_Vtt               = []
MC_bunchTimes        = []
MC_External          = []
MC_FV                = []
MC_NC                = []
MC_PDG               = []
MC_file_index        = []   # which file each cluster came from
MC_event_number      = []
MC_MRDT = []

# enable for alternative CC sideband
if ALTERNATIVE_SIDEBAND:
    MC_hitZ  = []
    MC_hitPE = []
    MC_MRD_hitT  = []
    
if WEIGHTING:
    MC_CV_weight = []


for file_idx in trange(len(file_names), desc='Loading MC files'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    MC_cluster_number.append(T['clusterNumber'].array(library='np'))
    MC_cluster_charge.append(T['clusterPE'].array(library='np'))
    MC_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    MC_cluster_Hits.append(T['clusterHits'].array(library='np'))
    MC_cluster_time.append(T['clusterTime'].array(library='np'))
    MC_bunchTimes.append(T['bunchTimes'].array(library='np'))
    MC_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    MC_NoVeto.append(T['NoVeto'].array(library='np'))
    MC_event_number.append(CEN)
    MC_file_index.append(np.full(len(CEN), file_idx, dtype=np.int32))
    
    if ALTERNATIVE_SIDEBAND:
        MC_hitZ.append(T['hitZ'].array())
        MC_hitPE.append(T['hitPE'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')

    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2
    ext_flags = (r2 > radius**2)    | (np.abs(vy_cm) > half_height)

    # map trigger-level quantities onto clusters via CEN
    MC_Vtx.append(vx_cm[CEN])
    MC_Vty.append(vy_cm[CEN])
    MC_Vtz.append(vz_cm[CEN])
    MC_Vtt.append(S['trueNuIntxVtx_T'].array(library='np')[CEN])
    MC_External.append(ext_flags[CEN])
    MC_NC.append(S['trueNC'].array(library='np')[CEN])
    MC_PDG.append(S['trueNuPDG'].array(library='np')[CEN])
    MC_MRDT.append(S['numMRDTracks'].array(library='np')[CEN])
    
    if WEIGHTING:
        MC_CV_weight.append(S['weight_TunedCentralValue_UBGenie'].array(library='np')[CEN])
    
    if ALTERNATIVE_SIDEBAND:
        mrd_t   = S['MRDhitT'].array()
        MC_MRD_hitT.append(mrd_t[CEN])
        

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

MC_cluster_number  = np.concatenate(MC_cluster_number)
MC_cluster_time    = np.concatenate(MC_cluster_time)
MC_cluster_charge  = np.concatenate(MC_cluster_charge)
MC_cluster_Qb      = np.concatenate(MC_cluster_Qb)
MC_cluster_Hits    = np.concatenate(MC_cluster_Hits)
MC_bunchTimes      = np.concatenate(MC_bunchTimes)
MC_TankMRDCoinc    = np.concatenate(MC_TankMRDCoinc)
MC_NoVeto          = np.concatenate(MC_NoVeto)
MC_event_number    = np.concatenate(MC_event_number)
MC_file_index      = np.concatenate(MC_file_index)
MC_Vtx             = np.concatenate(MC_Vtx)
MC_Vty             = np.concatenate(MC_Vty)
MC_Vtz             = np.concatenate(MC_Vtz)
MC_Vtt             = np.concatenate(MC_Vtt)
MC_External        = np.concatenate(MC_External)
MC_NC              = np.concatenate(MC_NC)
MC_PDG             = np.concatenate(MC_PDG)
MC_MRDT            = np.concatenate(MC_MRDT)

if ALTERNATIVE_SIDEBAND:
    MC_hitZ     = ak.concatenate(MC_hitZ)
    MC_hitPE    = ak.concatenate(MC_hitPE)
    MC_MRD_hitT = ak.concatenate(MC_MRD_hitT)
    
if WEIGHTING:
    MC_CV_weight = np.concatenate(MC_CV_weight)


print(f'POT: {POT:.3e}')
print(f'Total clusters: {len(MC_cluster_number)}')
print('\ndone')

In [ ]:
# data       

def extract_run_number(file_name):
    return int(file_name.split('_')[0][1:])  # Split by '_' and remove 'R' to get the run number

directory = 'FY22_23/'
file_names = [file for file in os.listdir(directory) if os.path.isfile(os.path.join(directory, file))]
print('There are: ', len(file_names), ' files\n')

# POT file
csv_file = 'POT_files/POT_trigger_FY22_23_summary.csv'
pot_df = pd.read_csv(csv_file)

# extract total POT and triggers
total_POT_TOR860 = sum(pot_df["TOR860_POTe12"])/(1e8)   # convert to e20 POT
total_triggers = sum(pot_df["TOTAL_TRIGGERS"])

cluster_time = []
cluster_charge = []
cluster_Hits = []
cluster_Qb = []
TankMRDCoinc = []
NoVeto = []
MRD_tracks = []
isBrightest = []
cluster_Number = []
runs = []

if ALTERNATIVE_SIDEBAND:
    MRD_activity = []
    hitZ = []
    hitPE = []

counter = 0
print('(', (counter), '/', len(file_names), ')')

for file_name in file_names:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 25 == 0:
        print('(', (counter), '/', len(file_names), ')')
    
    with uproot.open(os.path.join(directory, file_name)) as file:
        
        Event = file["data"]
        
        runs.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time.append(Event["cluster_time_BRF"].array(library="np"))
        cluster_Number.append(Event["cluster_Number"].array(library="np"))
        cluster_charge.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto.append(Event["NoVeto"].array(library="np"))
        MRD_tracks.append(Event["MRD_Track"].array(library="np"))
        isBrightest.append(Event["isBrightest"].array(library="np"))
        
        if ALTERNATIVE_SIDEBAND:
            MRD_activity.append(Event["MRD_activity"].array(library="np"))
            hitZ.append(Event["hitZ"].array())
            hitPE.append(Event["hitPE"].array())
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs = np.concatenate(runs)
cluster_time = np.concatenate(cluster_time)
cluster_Number = np.concatenate(cluster_Number)
cluster_charge = np.concatenate(cluster_charge)
cluster_Qb = np.concatenate(cluster_Qb)
cluster_Hits = np.concatenate(cluster_Hits)
TankMRDCoinc = np.concatenate(TankMRDCoinc)
NoVeto = np.concatenate(NoVeto)
MRD_tracks = np.concatenate(MRD_tracks)
isBrightest = np.concatenate(isBrightest)

if ALTERNATIVE_SIDEBAND:
    MRD_activity = np.concatenate(MRD_activity)
    hitZ = ak.concatenate(hitZ)
    hitPE = ak.concatenate(hitPE)

print(total_triggers, 'triggers')
print(total_POT_TOR860, 'e20 POT\n')

In [ ]:
# off beam data

directory_ext = 'off_beam/'
file_names_ext = [file for file in os.listdir(directory_ext) if os.path.isfile(os.path.join(directory_ext, file))]
print('There are: ', len(file_names_ext), ' files\n')

# POT file
csv_file_ext = 'POT_files/offbeam_summary.csv'
pot_df_ext = pd.read_csv(csv_file_ext)

# extract total POT and triggers
total_ext_triggers = sum(pot_df_ext["TOTAL_TRIGGERS"])

cluster_time_ext = []
cluster_charge_ext = []
cluster_Hits_ext = []
cluster_Qb_ext = []
TankMRDCoinc_ext = []
NoVeto_ext = []
MRD_tracks_ext = []
isBrightest_ext = []
cluster_Number_ext = []
runs_ext = []

if ALTERNATIVE_SIDEBAND:
    MRD_activity_ext = []
    hitZ_ext = []
    hitPE_ext = []

counter = 0
print('(', (counter), '/', len(file_names_ext), ')')

for file_name in file_names_ext:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 25 == 0:
        print('(', (counter), '/', len(file_names), ')')
    
    with uproot.open(os.path.join(directory_ext, file_name)) as file:
        
        Event = file["data"]
        
        runs_ext.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time_ext.append(Event["cluster_time"].array(library="np"))
        cluster_Number_ext.append(Event["cluster_Number"].array(library="np"))
        cluster_charge_ext.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb_ext.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits_ext.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc_ext.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto_ext.append(Event["NoVeto"].array(library="np"))
        MRD_tracks_ext.append(Event["MRD_Track"].array(library="np"))
        isBrightest_ext.append(Event["isBrightest"].array(library="np"))
        
        if ALTERNATIVE_SIDEBAND:
            MRD_activity_ext.append(Event["MRD_activity"].array(library="np"))
            hitZ_ext.append(Event["hitZ"].array())
            hitPE_ext.append(Event["hitPE"].array())
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs_ext = np.concatenate(runs_ext)
cluster_time_ext = np.concatenate(cluster_time_ext)
cluster_Number_ext = np.concatenate(cluster_Number_ext)
cluster_charge_ext = np.concatenate(cluster_charge_ext)
cluster_Qb_ext = np.concatenate(cluster_Qb_ext)
cluster_Hits_ext = np.concatenate(cluster_Hits_ext)
TankMRDCoinc_ext = np.concatenate(TankMRDCoinc_ext)
NoVeto_ext = np.concatenate(NoVeto_ext)
MRD_tracks_ext = np.concatenate(MRD_tracks_ext)
isBrightest_ext = np.concatenate(isBrightest_ext)

if ALTERNATIVE_SIDEBAND:
    MRD_activity_ext = np.concatenate(MRD_activity_ext)
    hitZ_ext = ak.concatenate(hitZ_ext)
    hitPE_ext = ak.concatenate(hitPE_ext)

    
print(total_ext_triggers, 'triggers')

In [ ]:
# make MC cuts

# numuCC, nueCC, other (including nubar), NC, external
MC_PE = [[], [], [], [], []]
MC_bunch_times = [[], [], [], [], []]
MC_hits = [[], [], [], [], []]

MC_weights_by_cat = [[], [], [], [], []]  # if weighting


for i in trange(len(MC_cluster_time), desc = 'applying CCinc selection cuts...'):
    
    if MC_TankMRDCoinc[i] == 0:   # must have Tank/MRD coincidence
        continue
    if MC_MRDT[i] != 1:          # not a single track
        continue
        
    if MC_NoVeto[i] == 0:
        continue
        
    if MC_bunchTimes[i] > bunch_time_cutoff:
        continue
    
    if MC_cluster_Qb[i] > 0.2 or MC_cluster_Qb[i] < 0:
        continue
        
    if MC_cluster_number[i] != 0:
        continue
        
    brightest = MC_cluster_charge[i]; flag = False
    for j in range(i+1, len(MC_cluster_number)):
        if MC_cluster_number[j] == 0:
            break
        if MC_cluster_charge[j] > brightest:
            flag = True
            break
    if flag == True:
        continue
    
    if MC_cluster_Hits[i] < 100:
        continue
        
    # ALTERNATIVE SIDEBAND: instead of MRD coincidence, require the charge barycenter to be downstream and for there to be ANY MRD activity
    #if not any(0 <= t <= 300 for t in MC_MRD_hitT[i]):
    #    continue
    if ALTERNATIVE_SIDEBAND:
        a = 0
        for j in range(len(MC_hitPE[i])):
            a += MC_hitPE[i][j]*MC_hitZ[i][j]
        if a < 0:
            continue
        
        
    # event categorization
    if MC_External[i] == True:
        indy = 4
        MC_PE[indy].append(MC_cluster_charge[i])
        MC_hits[indy].append(MC_cluster_Hits[i])
        MC_bunch_times[indy].append(MC_bunchTimes[i])
        MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
        
    else:
        if MC_NC[i] == 0:         # CC
            if MC_PDG[i] == 14:   # numuCC
                indy = 0
                MC_PE[indy].append(MC_cluster_charge[i])
                MC_hits[indy].append(MC_cluster_Hits[i])
                MC_bunch_times[indy].append(MC_bunchTimes[i])
                MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
            elif MC_PDG[i] == 12:    # nueCC
                indy = 1
                MC_PE[indy].append(MC_cluster_charge[i])
                MC_hits[indy].append(MC_cluster_Hits[i])
                MC_bunch_times[indy].append(MC_bunchTimes[i])
                MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
            else:    # other (nubar CC)
                indy = 2
                MC_PE[indy].append(MC_cluster_charge[i])
                MC_hits[indy].append(MC_cluster_Hits[i])
                MC_bunch_times[indy].append(MC_bunchTimes[i])
                MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
        else:    # NC
            indy = 3
            MC_PE[indy].append(MC_cluster_charge[i])
            MC_hits[indy].append(MC_cluster_Hits[i])
            MC_bunch_times[indy].append(MC_bunchTimes[i])
            MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
          

print('done')

In [ ]:
for i in range(len(MC_bunch_times)):
    print(len(MC_bunch_times[i]))

In [ ]:
bins = np.arange(0,1800,2)
plt.hist(MC_bunch_times[0], bins = bins)
plt.hist(MC_bunch_times[4], bins = bins, histtype = 'step', zorder = 2)
plt.show()

In [ ]:
# data

PE = []
bunch_times = []
hits = []
reco_runs = []


for i in trange(len(cluster_time), desc = 'applying CCinc selection cuts to data...'):
    
    if(MRD_tracks[i]==0):    # 1 track
        continue
    if(TankMRDCoinc[i]==0):  # TankMRDCoinc
        continue
    if(NoVeto[i]==0):   # NoVeto
        continue
    if(isBrightest[i]==0):    # Brightest
        continue
    if(cluster_Qb[i]>0.2 or cluster_Qb[i]<0): # Cluster Charge Balance < 0.2
        continue
    if(cluster_Hits[i]<100):   # indicative the muon topology is not center-aligned
        continue
        
    # Prompt (first) cluster
    if cluster_Number[i] != 0:
        continue
        
    if runs[i] > 4000:   # FY23
        if cluster_time[i] < spill_start_23:
            continue
        if cluster_time[i] > spill_end_23:
            continue
    else:
        if cluster_time[i] < spill_start_22:
            continue
        if cluster_time[i] > spill_end_22:
            continue
            
    
    # ALTERNATE sideband: any MRD activity (any hits) + downstream charge
    #if MRD_activity[i] == False:
    #    continue
    if ALTERNATIVE_SIDEBAND:
        a = 0
        for j in range(len(hitPE[i])):
            a += hitPE[i][j]*hitZ[i][j]
        if a < 0:
            continue
    
        
    # event selection
    PE.append(cluster_charge[i])
    hits.append(cluster_Hits[i])
    bunch_times.append(cluster_time[i])
    reco_runs.append(runs[i])
          

print('done')

In [ ]:
print(len(bunch_times))

In [ ]:
# ext data

# Importantly off-beam data does NOT have the MRD
# this is a flaw in all ANNIE analyses and appropriate systematics need to be applied
# Based on the pre-spill structure, the event rate of the off-beam / cosmics is damn near negligible though
# so it can be approximated as zero for now

ext_PE = []
ext_bunch_times = []
ext_hits = []
ext_reco_runs = []

for i in trange(len(cluster_time_ext), desc = 'applying CCinc selection cuts to off-beam data...'):
    
    if(MRD_tracks_ext[i]==0):    # 1 track
        continue
    if(TankMRDCoinc_ext[i]==0):  # TankMRDCoinc
        continue
    if(NoVeto_ext[i]==0):   # NoVeto
        continue
    if(isBrightest_ext[i]==0):    # Brightest
        continue
    if(cluster_Qb_ext[i]>0.2 or cluster_Qb_ext[i]<0): # Cluster Charge Balance < 0.2
        continue
    if(cluster_Hits_ext[i]<100):   # indicative the muon topology is not center-aligned
        continue
        
    # Prompt (first) cluster
    if cluster_Number_ext[i] != 0:
        continue
        
    if runs_ext[i] > 4000:   # FY23
        if cluster_time_ext[i] < spill_start_23:
            continue
        if cluster_time_ext[i] > spill_end_23:
            continue
    else:
        if cluster_time_ext[i] < spill_start_22:
            continue
        if cluster_time_ext[i] > spill_end_22:
            continue
            
    # alternative sideband
    #if MRD_activity_ext[i] == False:
    #    continue
    if ALTERNATIVE_SIDEBAND:
        a = 0
        for j in range(len(hitPE_ext[i])):
            a += hitPE_ext[i][j]*hitZ_ext[i][j]
        if a < 0:
            continue
        
    # event selection
    ext_PE.append(cluster_charge_ext[i])
    ext_hits.append(cluster_Hits_ext[i])
    ext_bunch_times.append(cluster_time_ext[i])
    ext_reco_runs.append(runs_ext[i])
          

print('done')

In [ ]:
# plotting function for PMT observables

def plot_sideband_comparison(bins, mc_data, on_beam_data, off_beam_data, 
                             pot_mc, pot_data, trig_ext, trig_data,
                             axis_label, y_axis_label, x_limit, y_limit, xaxis_minor, 
                             plot_name, ext_scale_factor=1.0, legend_columns=3, mc_weights_by_cat=None):
    
    # 1. Scaling Factors
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext
    
    # 2. Prepare Prediction Stack (MC + Off-Beam)
    # Order: [Ext_MC, OOF, CC, NCother, nubarNCQE, nuNCQE, Off-Beam Data]
    prediction_data = mc_data + [off_beam_data]
    
    # --- MC weights including GENIE tuning weights ---
    mc_weights = []
    for i, arr in enumerate(mc_data):
        if mc_weights_by_cat is None:
            genie_w = np.ones(len(arr))
        else:
            genie_w = np.array(mc_weights_by_cat[i])
        mc_weights.append(genie_w * f_pot)

    # Assemble full weight list
    weights = mc_weights + [np.ones(len(off_beam_data)) * f_ext]
    
    #weights = [np.ones(len(arr)) * f_pot for arr in mc_data]
    #weights.append(np.ones(len(off_beam_data)) * f_ext)
    
    # --- DEBUG PRINTS ---
    #print(f"f_pot scaling factor: {f_pot:.3f}")
    #print(f"Raw Dirt events before weights: {len(prediction_data[0])}")
    #print(f"Dirt Yield BEFORE scale: {np.sum(weights[0]):.1f}")
    
    # 3. Aesthetics
    labels_base = ['External (MC)', r'$\nu_{\mu}$' + 'CC', r'$\nu_{e}$' + 'CC', 'NC', 'Other', 'Off-Beam Data']
    colors = ['papayawhip', '#0066ff', 'cyan', 'darkred', 'darkorange', 'gray']
    # Match outline colors to the fill (or use 'navy' as in your original script)
    color_outline = ['navy'] * 5 + ['black']
    
    for i, label in enumerate(labels_base):
        if 'External (MC)' in label:
            weights[i] = weights[i] * ext_scale_factor  # <--- CHANGED HERE
            print(f"Applied {ext_scale_factor:.4f} scale to index {i} ({label})")
            
    #print(f"Dirt Yield AFTER scale: {np.sum(weights[0]):.1f}")
    #print(f"Off-Beam Yield: {np.sum(weights[-1]):.1f}")

    # Calculate scaled yields for labels
    yields = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1
    labels = [f"{l} ({100*y/total_pred_yield:.1f}%)" for l, y in zip(labels_base, yields)]
    
    # 4. Math for Residuals and Errors
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths = np.diff(bins)
    
    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    
    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0] for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)
    
    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0] for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))
    
    # ---------------------------------------------------------
    # 5. Calculate Chi-Squared
    # ---------------------------------------------------------
    # Total variance in each bin: sigma_data^2 + sigma_mc^2
    variance = (data_err ** 2) + (pred_err ** 2)
    
    # Mask out bins where both data and MC are 0 to avoid division by zero
    valid_bins = variance > 0
    
    # Calculate chi^2 using only the valid bins
    chi2 = np.sum(((data_counts[valid_bins] - pred_counts[valid_bins]) ** 2) / variance[valid_bins])
    
    # Degrees of freedom: Number of fitted bins minus 1 (for the normalization constraint)
    ndf = np.sum(valid_bins) - 1
    
    print(f"\nExternal Scale Factor Used: {ext_scale_factor:.4f}")
    if ndf > 0:
        print(f"Chi2/NDF: {chi2:.2f} / {ndf} = {chi2 / ndf:.2f}")
    else:
        print("Chi2/NDF: Error - Not enough populated bins.")
    # ---------------------------------------------------------

    # --- Plotting ---
    fig = plt.figure(figsize=(7, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.1)
    ax = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # A. The Fill (No edges to avoid the bar-graph look)
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            label=labels, color=colors, linewidth=0)

    # B. The Outline (Reverting to your original 'step' logic for clean category boundaries)
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            histtype='step', color=color_outline, linewidth=0.8)

    # C. On-Beam Data Points
    ax.errorbar(bin_centers, data_counts, yerr=data_err, xerr=bin_widths/2, 
                fmt='ko', label='On-Beam Data', markersize=4, capsize=0)

    # --- Residual Plot: (Data - Prediction) / Prediction ---
    mask = pred_counts > 0
    res = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)
    
    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    
    # Error propagation for (D/P - 1)
    # sigma_res = (D/P) * sqrt( (sigD/D)^2 + (sigP/P)^2 )
    ratio = data_counts[mask] / pred_counts[mask]
    # Handle division by zero for error calculation if data_counts is 0
    d_term = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask], xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
    
    # Formatting
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=14)
    ax2.set_xlabel(axis_label)
    ax2.set_xlim(x_limit)
    ax2.set_ylim([-1.2, 1.2]) # Centered on 0
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax.legend(ncol=legend_columns, fontsize=9, loc='upper right', frameon=False)
    
    ax.text(0.02, 1.02, 'ANNIE', fontweight='bold', color='blue', transform=ax.transAxes)
    ax.text(0.17, 1.02, 'Preliminary', transform=ax.transAxes)
    ax.text(0.70, 1.02, f"{pot_data:.2f} " + r"$\times 10^{20}$ POT", transform=ax.transAxes)
    
    # chi^2 stat
    ax.text(0.6, 0.75, r"$\chi^2 / \text{dof}$ = " + f"{chi2:.1f} / {ndf}",
            transform=ax.transAxes, fontsize = 12)

    plt.tight_layout()
    plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()


print('Plotting function loaded')

In [ ]:
# --- 1. Scaling Constants ---
POT_MC = POT/(1e20)
POT_DATA = total_POT_TOR860
TRIG_DATA = total_triggers
TRIG_EXT = total_ext_triggers

#['External (MC)', r'$\nu_{\mu}$' + 'CC', r'$\nu_{e}$' + 'CC', 'NC', 'Other', 'Off-Beam Data']
# numuCC, nueCC, other (including nubar), NC, external
# --> correct ordering is [4], [0], [1], [3], [2]

print('done')

In [ ]:
def plot_all_observables(PMT_hits=False):
    
    path_folder = 'plots/sideband/CCinc/genie_tuning/'
    os.system('mkdir -p ' + path_folder)
    
    mc_weights_stack = [
        MC_weights_by_cat[4],
        MC_weights_by_cat[0],
        MC_weights_by_cat[1],
        MC_weights_by_cat[3],
        MC_weights_by_cat[2]
    ]
    
    # Common arguments for the plotting function to avoid repetition
    common_args = {
        'pot_mc': POT_MC,
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 2,
        'mc_weights_by_cat': mc_weights_stack
        
    }

    # 2. PMT Hits
    if PMT_hits:
        print('\nPlotting PMT hits...\n')
        mc_stack = [MC_hits[4], MC_hits[0], MC_hits[1], MC_hits[3], MC_hits[2]]
        
        plot_sideband_comparison(
            bins=np.arange(100, 133, 1),
            mc_data=mc_stack,
            on_beam_data=hits,
            off_beam_data=ext_hits,
            axis_label='PMT hits',
            y_axis_label='Events / 4 hit bin',
            x_limit=[100, 132],
            y_limit=[0,5000],
            xaxis_minor=1,
            plot_name=path_folder + 'CCinc_sideband_PMT_hits.png',
            **common_args
        )

    return


plot_all_observables(PMT_hits=True)
print('\nAll sideband plots generated')

In [ ]:
centers = 29.250575891023246
centers_data_23 = 198.3700379
centers_data_22 = 215.7494731

def get_normalized_times(times, runs, mc=False):
    norm_times = []
    for t, r in zip(times, runs):
        if mc:
            t0 = centers
        else:
            t0 = centers_data_22 if r < 4000 else centers_data_23
        
        tnorm = t - t0
        # Filter to only include the 80 bunches (approx 18.936ns * 80)
        if 0 <= tnorm < (80 * 18.936):
            norm_times.append(tnorm)
    return np.array(norm_times)

# 1. Process MC
bunches_80_mc = [get_normalized_times(MC_bunch_times[i], [0]*len(MC_bunch_times[i]), mc=True) 
                 for i in range(5)]

# 2. Process On-Beam Data
bunches_80_data = get_normalized_times(bunch_times, reco_runs)

# 3. Process Off-Beam (External)
bunches_80_ext = get_normalized_times(ext_bunch_times, ext_reco_runs)

print('done')

In [ ]:
# weights

def get_normalized_times(times, runs, weights=None, mc=False):

    norm_times = []
    norm_weights = []

    for i, (t, r) in enumerate(zip(times, runs)):

        if mc:
            t0 = centers
        else:
            t0 = centers_data_22 if r < 4000 else centers_data_23

        tnorm = t - t0

        # Filter to only include the 80 bunches
        if 0 <= tnorm < (80 * 18.936):

            norm_times.append(tnorm)

            if weights is not None:
                norm_weights.append(weights[i])

    if weights is not None:
        return np.array(norm_times), np.array(norm_weights)

    return np.array(norm_times)


# 1. Process MC (Categories 0-5)
bunches_80_mc = []
bunches_80_weights = []

for i in range(5):

    t, w = get_normalized_times(
        MC_bunch_times[i],
        [0]*len(MC_bunch_times[i]),
        weights=MC_weights_by_cat[i],
        mc=True
    )

    bunches_80_mc.append(t)
    bunches_80_weights.append(w)

# 2. Process On-Beam Data
bunches_80_data = get_normalized_times(bunch_times, reco_runs)

# 3. Process Off-Beam (External)
bunches_80_ext = get_normalized_times(ext_bunch_times, ext_reco_runs)

print('done')

In [ ]:
# Standard BNB bunch spacing is ~18.936 ns
# 8 bunches per bin * 10 bins = 80 bunches
bin_width = 18.936 * 8  # Buckets of 16 bunches as per your request
bins = np.arange(0, 18.936 * 81, bin_width)
#bins = np.arange(0, 18.936 * 26, bin_width)

# Reorder MC for the stack ([4], [0], [1], [3], [2])
mc_stack = [bunches_80_mc[4], bunches_80_mc[0], bunches_80_mc[1], 
            bunches_80_mc[3], bunches_80_mc[2]]

mc_weights_stack = [
    bunches_80_weights[4],
    bunches_80_weights[0],
    bunches_80_weights[1],
    bunches_80_weights[3],
    bunches_80_weights[2]
]

common_args = {
        'pot_mc': POT_MC,
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 3,
        'mc_weights_by_cat': mc_weights_stack
    }

plot_sideband_comparison(
    bins=bins,
    mc_data=mc_stack,
    on_beam_data=bunches_80_data,
    off_beam_data=bunches_80_ext,
    axis_label='Time since first bunch [ns]',
    y_axis_label='Events / 8 BNB bunches',
    x_limit=[0, 18.936 * 80],
    y_limit=[0,10000],
    xaxis_minor=50,
    plot_name='plots/sideband/CCinc/genie_tuning/CCinc_genie_tuning_CC_scaling.png',
    **common_args
)

print('done')

In [ ]:
# ... (Your existing bin definitions and mc_stack logic) ...

#for i in range(30):
    #scale_factor = 0.05 + i/100
    #print(scale_factor)
common_args = {
        'pot_mc': POT_MC,
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'ext_scale_factor': 0.1,#scale_factor, # <--- PASS IT HERE
        'legend_columns': 3,
        'mc_weights_by_cat': mc_weights_stack
    }

plot_sideband_comparison(
    bins=bins,
    mc_data=mc_stack,
    on_beam_data=bunches_80_data,
    off_beam_data=bunches_80_ext,
    axis_label='Time since first bunch [ns]',
    y_axis_label='Events / 8 BNB bunches',
    x_limit=[0, 18.936 * 80],
    y_limit=[0,7000],
    xaxis_minor=50,
    plot_name='plots/sideband/CCinc/genie_tuning/CCinc_bunch_dirt_scaling_TEST.png',
    **common_args
)

print('\ndone')

In [ ]:
# simple event rate bin

# ---------------------------------------------------------
# 1D RATE NORMALIZATION (Total Yield Matching)
# ---------------------------------------------------------
f_pot = POT_DATA / POT_MC
f_ext = TRIG_DATA / TRIG_EXT

# Total Data Events
total_data = len(bunches_80_data)

# Total Fixed Backgrounds (MC + Off-Beam)
# Note: mc_stack[0] is External MC, so we sum from index 1 onward
total_fixed_mc = sum([len(mc_stack[i]) * f_pot for i in range(1, len(mc_stack))])
total_off_beam = len(bunches_80_ext) * f_ext
total_fixed = total_fixed_mc + total_off_beam

# Total Unscaled External MC
total_ext_mc = len(mc_stack[0]) * f_pot

# Solve for the scale factor alpha
rate_scale_factor = (total_data - total_fixed) / total_ext_mc

print("\n" + "="*45)
print(" 1D RATE NORMALIZATION")
print("="*45)
print(f"Total Data Events:         {total_data}")
print(f"Total Fixed Backgrounds:   {total_fixed:.1f}")
print(f"Total Unscaled Ext MC:     {total_ext_mc:.1f}")
print("-" * 45)
print(f"Calculated Scale Factor:   {rate_scale_factor:.4f}")
print("="*45 + "\n")